# BDC 2026 - Kalibrasi Threshold Per-Kelas (Post-Processing, TANPA Training Ulang)

Berdasarkan temuan similarity-check: training set tidak punya contoh untuk pola "kerajinan
menyerupai organik" / "foto kemasan menampilkan isi produk", jadi solusinya bergeser dari ranah
training ke ranah **kalibrasi output**.

**Ide dasarnya:** alih-alih ambil kelas dengan probabilitas tertinggi apa adanya (`argmax`), kita
cari **bobot per-kelas** yang mengalikan probabilitas sebelum `argmax` -- dioptimasi supaya macro F1
naik. Ini bisa mengoreksi bias sistematis (mis. model "terlalu mudah" pilih Organic) tanpa menyentuh
model atau data training sama sekali.

**Supaya tidak sekadar overfitting ke `solution.csv`:** data itu dibagi 2 -- separuh buat mencari
bobot optimal (tuning set), separuh lagi buat validasi apakah perbaikannya generalisasi (holdout set),
bukan cuma "hafal" 1.458 data itu.

**Pakai submission juara** (Stacking ConvNeXt+SigLIP2 v2) sebagai basis -- cache probabilitasnya
sudah ada (`test_probs_partial.npy` + `meta_model.pkl`), jadi tidak perlu training/inference ulang.

In [1]:
import os

import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split

## Config

**PENTING:** cek dulu `meta_model_path` ini benar mengarah ke meta-learner juara (Stacking
ConvNeXt+SigLIP2 v2, macro F1 0,9799 di ranking kamu), bukan versi lain. Kalau kamu simpan dengan
nama file berbeda, sesuaikan di sini.

In [2]:
CONFIG = {
    "root": "BDC 2026",
    "solution_csv": "solution.csv",
    "submission_template": "BDC 2026/submission.csv",
    "meta_model_path": "meta_model.pkl",           # <- pastikan ini meta-learner juara (v2)
    "test_probs_partial_path": "test_probs_partial.npy",  # cache test probs [convnextv2_tiny, siglip2]
    "submission_out": "submission_champion_calibrated.csv",
    "holdout_frac": 0.5,   # separuh solution.csv buat cari bobot, separuh buat validasi
    "seed": 42,
}

## Deteksi Kelas & Test DataFrame

In [3]:
def get_class_order(config):
    train_dir = os.path.join(config["root"], "train")
    return sorted(os.listdir(train_dir))


def build_test_dataframe(config):
    test_dir = os.path.join(config["root"], "test")
    test_images = sorted(
        os.listdir(test_dir),
        key=lambda x: int("".join(filter(str.isdigit, x)))
    )
    test_df = pd.DataFrame({"image": test_images})
    test_df["id"] = test_df["image"].apply(lambda x: int("".join(filter(str.isdigit, x))))
    return test_df


class_order = get_class_order(CONFIG)
print("Label mapping:", {name: i for i, name in enumerate(class_order)})

test_df = build_test_dataframe(CONFIG)

Label mapping: {'0_Recyclable': 0, '1_Electronic': 1, '2_Organic': 2}


## Hitung Probabilitas Test dari Meta-Learner Juara

Bukan cuma prediksi akhirnya (`predict`), tapi probabilitasnya (`predict_proba`) -- ini yang
dibutuhkan untuk kalibrasi.

In [4]:
meta_model = joblib.load(CONFIG["meta_model_path"])

test_probs_partial = np.load(CONFIG["test_probs_partial_path"], allow_pickle=True)
test_probs_list = [np.asarray(p, dtype=np.float64) for p in test_probs_partial]

X_meta_test = np.concatenate(test_probs_list, axis=1)
probs_test = meta_model.predict_proba(X_meta_test)  # shape (n_test, 3)

print(f"Shape probabilitas test: {probs_test.shape}")
print(f"Model classes_: {meta_model.classes_}")

Shape probabilitas test: (1458, 3)
Model classes_: [0 1 2]


## Gabungkan dengan solution.csv, Split Tuning vs Holdout

In [5]:
assert os.path.exists(CONFIG["solution_csv"]), (
    f'Tidak ketemu \'{CONFIG["solution_csv"]}\' di direktori kerja saat ini '
    f'({os.getcwd()}). Cek lagi apakah file ini ada persis di folder itu.'
)
gt = pd.read_csv(CONFIG["solution_csv"])
assert "predicted" in gt.columns and "id" in gt.columns, (
    f"solution.csv kebaca, tapi kolomnya tidak sesuai ekspektasi (id, predicted). "
    f"Kolom yang ada sekarang: {gt.columns.tolist()}"
)
gt["predicted"] = gt["predicted"].fillna(0).astype(int)
gt = gt.rename(columns={"predicted": "true_label"})

eval_df = test_df[["id"]].copy()
eval_df["row_idx"] = np.arange(len(test_df))
eval_df = eval_df.merge(gt, on="id", how="inner")

y_true_all = eval_df["true_label"].values
probs_all = probs_test[eval_df["row_idx"].values]

print(f"Total baris dengan label manual: {len(eval_df)}")

tuning_idx, holdout_idx = train_test_split(
    np.arange(len(eval_df)),
    test_size=CONFIG["holdout_frac"],
    stratify=y_true_all,
    random_state=CONFIG["seed"],
)

y_tuning, probs_tuning = y_true_all[tuning_idx], probs_all[tuning_idx]
y_holdout, probs_holdout = y_true_all[holdout_idx], probs_all[holdout_idx]

print(f"Tuning set : {len(tuning_idx)} baris")
print(f"Holdout set: {len(holdout_idx)} baris")

Total baris dengan label manual: 1458
Tuning set : 729 baris
Holdout set: 729 baris


## Optimasi Bobot Per-Kelas (Grid Search)

`adjusted_probs = probs * w` sebelum `argmax`. Dicari `w` yang memaksimalkan macro F1 di TUNING set
saja (holdout belum dipakai di sini) -- lewat grid search 2D (bukan Nelder-Mead, lihat catatan di
cell kode).

In [6]:
def macro_f1_with_weights(w, probs, y_true):
    adjusted = probs * w
    preds = adjusted.argmax(axis=1)
    return f1_score(y_true, preds, average="macro")


baseline_f1_tuning = macro_f1_with_weights(np.array([1.0, 1.0, 1.0]), probs_tuning, y_tuning)
print(f"Macro F1 SEBELUM kalibrasi (tuning set): {baseline_f1_tuning:.4f}")

# Grid search, bukan Nelder-Mead -- dijamin menemukan titik terbaik di ruang pencarian
# ini, tidak bisa "macet" seperti gradient-free optimizer di fungsi yang tidak mulus
# (macro F1 dari argmax itu berubah loncat-loncat, bukan halus, jadi Nelder-Mead gampang
# berhenti di titik yang cuma versi diskalakan rata dari [1,1,1] -- yang argmax-nya
# IDENTIK dengan baseline, makanya F1 sebelum/sesudah sama persis kemarin).
#
# w1 (Electronic) dikunci di 1.0 karena pola errornya spesifik Recyclable<->Organic,
# bukan melibatkan Electronic. w0 (Recyclable) & w2 (Organic) digeser relatif satu sama lain.
w_range = np.linspace(0.05, 5.0, 120)  # dilebarkan dari 0.3-3.0 -- w_organic kemarin mentok di batas bawah

best_f1 = -1
best_weights = np.array([1.0, 1.0, 1.0])
for w0 in w_range:
    for w2 in w_range:
        w = np.array([w0, 1.0, w2])
        f1 = macro_f1_with_weights(w, probs_tuning, y_tuning)
        if f1 > best_f1:
            best_f1 = f1
            best_weights = w

print(f"\nBobot terbaik ditemukan: {dict(zip(class_order, best_weights.round(4)))}")
print(f"Macro F1 SESUDAH kalibrasi (tuning set): {best_f1:.4f}")

if np.allclose(best_weights / best_weights[1], np.array([1.0, 1.0, 1.0])):
    print("\n>> Bobot terbaik = [1,1,1] (tidak ada perubahan) -- tidak ada kalibrasi weight sederhana yang membantu.")


Macro F1 SEBELUM kalibrasi (tuning set): 0.9794

Bobot terbaik ditemukan: {'0_Recyclable': np.float64(0.6739), '1_Electronic': np.float64(1.0), '2_Organic': np.float64(0.05)}
Macro F1 SESUDAH kalibrasi (tuning set): 0.9892


## Validasi di Holdout Set

Ini pengecekan paling penting -- kalau perbaikan cuma muncul di tuning set tapi HILANG/memburuk di
holdout, berarti bobotnya cuma overfitting ke `solution.csv`, bukan perbaikan yang genuine.

In [7]:
baseline_f1_holdout = macro_f1_with_weights(np.array([1.0, 1.0, 1.0]), probs_holdout, y_holdout)
calibrated_f1_holdout = macro_f1_with_weights(best_weights, probs_holdout, y_holdout)

print(f"Macro F1 holdout SEBELUM kalibrasi: {baseline_f1_holdout:.4f}")
print(f"Macro F1 holdout SESUDAH kalibrasi: {calibrated_f1_holdout:.4f}")
print(f"Selisih                           : {calibrated_f1_holdout - baseline_f1_holdout:+.4f}")

if calibrated_f1_holdout > baseline_f1_holdout:
    print("\n>> Perbaikan TERKONFIRMASI di holdout set -- bukan cuma overfitting ke tuning set.")
else:
    print("\n>> PERINGATAN: tidak ada perbaikan (atau memburuk) di holdout set.")
    print("   Ini tanda kalibrasi ini overfitting ke solution.csv, JANGAN dipakai untuk submission akhir.")

print("\n=== Classification Report Holdout SEBELUM kalibrasi ===")
print(classification_report(y_holdout, probs_holdout.argmax(axis=1), target_names=class_order))
print("\n=== Classification Report Holdout SESUDAH kalibrasi ===")
print(classification_report(y_holdout, (probs_holdout * best_weights).argmax(axis=1), target_names=class_order))

print("\nConfusion matrix holdout SEBELUM:")
print(confusion_matrix(y_holdout, probs_holdout.argmax(axis=1)))
print("\nConfusion matrix holdout SESUDAH:")
print(confusion_matrix(y_holdout, (probs_holdout * best_weights).argmax(axis=1)))

Macro F1 holdout SEBELUM kalibrasi: 0.9804
Macro F1 holdout SESUDAH kalibrasi: 0.9836
Selisih                           : +0.0033

>> Perbaikan TERKONFIRMASI di holdout set -- bukan cuma overfitting ke tuning set.

=== Classification Report Holdout SEBELUM kalibrasi ===
              precision    recall  f1-score   support

0_Recyclable       0.99      0.95      0.97       280
1_Electronic       0.99      0.99      0.99       100
   2_Organic       0.96      1.00      0.98       349

    accuracy                           0.98       729
   macro avg       0.98      0.98      0.98       729
weighted avg       0.98      0.98      0.98       729


=== Classification Report Holdout SESUDAH kalibrasi ===
              precision    recall  f1-score   support

0_Recyclable       0.99      0.96      0.98       280
1_Electronic       0.99      0.99      0.99       100
   2_Organic       0.97      0.99      0.98       349

    accuracy                           0.98       729
   macro avg       

## Terapkan ke Seluruh Test Set & Buat Submission

**Hanya jalankan cell ini kalau holdout di atas menunjukkan perbaikan nyata.** Kalau tidak, jangan
pakai kalibrasi ini -- lebih baik pertahankan submission juara yang lama.

In [8]:
calibrated_probs_full = probs_test * best_weights
final_preds = calibrated_probs_full.argmax(axis=1)

pred_map = dict(zip(test_df["id"], final_preds))

submission = pd.read_csv(CONFIG["submission_template"])
submission["predicted"] = submission["id"].map(pred_map)
assert submission["predicted"].isna().sum() == 0, "Ada id yang tidak ter-mapping, cek ulang!"
submission["predicted"] = submission["predicted"].astype(int)

submission.to_csv(CONFIG["submission_out"], index=False)
print(f"Submission hasil kalibrasi disimpan ke: {CONFIG['submission_out']}")
print(submission["predicted"].value_counts())

print(f"\n=== Evaluasi FULL solution.csv (tuning + holdout digabung) ===")
print("F1 Macro SEBELUM kalibrasi:", f1_score(y_true_all, probs_all.argmax(axis=1), average="macro"))
print("F1 Macro SESUDAH kalibrasi:", f1_score(y_true_all, (probs_all * best_weights).argmax(axis=1), average="macro"))

Submission hasil kalibrasi disimpan ke: submission_champion_calibrated.csv
predicted
2    712
0    545
1    201
Name: count, dtype: int64

=== Evaluasi FULL solution.csv (tuning + holdout digabung) ===
F1 Macro SEBELUM kalibrasi: 0.9799091344303085
F1 Macro SESUDAH kalibrasi: 0.9864354028723182
